# M2 — Segmentación de Instancias de Vehículos (YOLO11-seg)
Entrena las **5 clases obligatorias**: `moto`, `auto`, `minibus`, `camion`, `bus`.

Flujo automático:
1. Descarga datasets de **Roboflow Universe** (incluyendo vehículos en segmentación y transporte público de Bolivia: minibuses/micros/flotas).
2. Amplía con imágenes reales de vehículos bolivianos desde Wikimedia Commons y genera máscaras poligonales con `YOLOE` para las imágenes que solo tenían cajas.
3. Sube el dataset fusionado a tu proyecto de Roboflow (`cristopher-diego-rodriguez-ayala/peaje-vehiculos-seg-6laub`), genera versión con aumentaciones y entrena `yolo11s-seg.pt`.

In [ ]:
!nvidia-smi -L || echo "Activa GPU T4 en Entorno de ejecucion > Cambiar tipo de entorno"
!pip install -q ultralytics==8.3.107 roboflow==1.1.60 pyyaml pandas matplotlib requests

In [ ]:
import os, glob, json, shutil, random, yaml, cv2, requests, numpy as np, torch
from pathlib import Path
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORKSPACE = 'cristopher-diego-rodriguez-ayala'
PROJECT_M2 = 'peaje-vehiculos-seg-6laub'

CLASES_OBJETIVO = ['moto', 'auto', 'minibus', 'camion', 'bus']
ID_OBJETIVO = {c: i for i, c in enumerate(CLASES_OBJETIVO)}
MAPA_SINONIMOS = {
    'moto': 'moto', 'motorcycle': 'moto', 'motorbike': 'moto', 'motociclista': 'moto', 'scooter': 'moto',
    'auto': 'auto', 'car': 'auto', 'sedan': 'auto', 'suv': 'auto', 'pickup': 'auto', 'taxi': 'auto',
    'minibus': 'minibus', 'minibús': 'minibus', 'micro': 'minibus', 'trufi': 'minibus', 'van': 'minibus', 'minivan': 'minibus',
    'camion': 'camion', 'camión': 'camion', 'truck': 'camion', 'cargo truck': 'camion', 'trailer': 'camion',
    'bus': 'bus', 'autobus': 'bus', 'flota': 'bus', 'buses': 'bus',
}

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY') or getpass('Pega tu Roboflow Private API Key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print('Dispositivo:', DEVICE, '| Workspace:', WORKSPACE)

## 1. Descargar datasets base de Roboflow Universe + ampliar con imágenes de Bolivia

In [ ]:
BASE_DL = Path('/content/m2_fuentes')
MERGED_COCO = Path('/content/m2_coco_upload')
MERGED_YOLO = Path('/content/m2_yolo_train')
for d in (BASE_DL, MERGED_COCO, MERGED_YOLO):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

# 1A. Dataset ya segmentado (car, truck, bus, motorbike)
ds_seg = rf.workspace('personal-dashboard').project('vehicle-road-segmentation').version(1).download('yolov11', location=str(BASE_DL / 'seg_base'))
# 1B. Datasets de transporte publico de Bolivia (minibus, micro, bus, motociclista)
ds_bo1 = rf.workspace('lab2e3').project('transporte-publico-bolivia').version(3).download('yolov8', location=str(BASE_DL / 'bolivia_1'))
ds_bo2 = rf.workspace('jaqueline-paola-coronel-condori').project('transporte-publico').version(1).download('yolov8', location=str(BASE_DL / 'bolivia_2'))

# 1C. Ampliar con fotos reales libres de Wikimedia Commons (minibuses, motos, camiones, buses)
WIKI_DIR = BASE_DL / 'wikimedia'
WIKI_DIR.mkdir(parents=True, exist_ok=True)
queries_wiki = [
    ('minibus', 'Minibus La Paz Bolivia'),
    ('bus', 'Bus Bolivia flota'),
    ('camion', 'Truck Bolivia highway'),
    ('moto', 'Motorcycle street'),
    ('auto', 'Sedan car front view'),
]
headers = {'User-Agent': 'IAM811-PeajeEducativo/1.0 (https://unifranz.edu.bo)'}
wiki_imgs = []
for clase_hint, q in queries_wiki:
    try:
        r = requests.get('https://commons.wikimedia.org/w/api.php', params={
            'action': 'query', 'generator': 'search', 'gsrsearch': q, 'gsrnamespace': 6,
            'gsrlimit': 12, 'prop': 'imageinfo', 'iiprop': 'url|mime', 'iiurlwidth': 960, 'format': 'json'
        }, headers=headers, timeout=15).json()
        pages = (r.get('query') or {}).get('pages') or {}
        for pid, pinfo in pages.items():
            ii = (pinfo.get('imageinfo') or [{}])[0]
            url = ii.get('thumburl') or ii.get('url')
            mime = ii.get('mime', '')
            if url and ('jpeg' in mime or 'png' in mime):
                resp = requests.get(url, headers=headers, timeout=15)
                if resp.status_code == 200 and len(resp.content) > 8000:
                    fp = WIKI_DIR / f'wiki_{clase_hint}_{pid}.jpg'
                    fp.write_bytes(resp.content)
                    wiki_imgs.append((fp, clase_hint))
    except Exception as e:
        print('Aviso Wikimedia', q, e)
print('Imagenes adicionales descargadas de Wikimedia:', len(wiki_imgs))

## 2. Unificar polígonos de segmentación para las 5 clases (`moto`, `auto`, `minibus`, `camion`, `bus`)

In [ ]:
yoloe = YOLOE('yoloe-11l-seg.pt')
PROMPTS_M2 = {
    'motorcycle': 'moto', 'motorbike': 'moto',
    'car': 'auto', 'sedan': 'auto', 'pickup truck': 'auto',
    'minibus': 'minibus', 'van': 'minibus', 'minivan': 'minibus',
    'truck': 'camion', 'cargo truck': 'camion',
    'bus': 'bus',
}
yoloe.set_classes(list(PROMPTS_M2.keys()), yoloe.get_text_pe(list(PROMPTS_M2.keys())))

for sp in ('train', 'valid'):
    (MERGED_YOLO / sp / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED_YOLO / sp / 'labels').mkdir(parents=True, exist_ok=True)
    (MERGED_COCO / sp).mkdir(parents=True, exist_ok=True)

def cargar_names(ds_root):
    y = yaml.safe_load((Path(ds_root) / 'data.yaml').read_text())
    n = y['names']
    return {i: name for i, name in enumerate(n)} if isinstance(n, list) else {int(k): v for k, v in n.items()}

registros = []  # lista de (img_path, split, [(cls_id, poly_norm_Nx2)])

# A) Desde seg_base (ya tiene poligonos reales)
names_seg = cargar_names(ds_seg.location)
for split_in in ('train', 'valid', 'test'):
    img_dir = Path(ds_seg.location) / split_in / 'images'
    if not img_dir.exists():
        continue
    sp_out = 'valid' if split_in in ('valid', 'test') else 'train'
    for img_p in sorted(img_dir.glob('*')):
        lbl_p = img_p.parent.parent / 'labels' / f'{img_p.stem}.txt'
        if not lbl_p.exists():
            continue
        polis = []
        for line in lbl_p.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            raw_cls = str(names_seg[int(float(parts[0]))]).lower()
            norm_cls = MAPA_SINONIMOS.get(raw_cls)
            if norm_cls is None:
                continue
            coords = np.array([float(x) for x in parts[1:]], dtype=float).reshape(-1, 2)
            polis.append((ID_OBJETIVO[norm_cls], coords))
        if polis:
            registros.append((img_p, sp_out, polis))

# B) Desde bolivia_1 y bolivia_2 (cajas -> refinamos poligono con YOLOE dentro o directo rectangulo si falla)
def extraer_polis_yoloe(im_bgr, filtro_cajas=None):
    H, W = im_bgr.shape[:2]
    r = yoloe.predict(im_bgr, conf=0.22, imgsz=640, device=DEVICE, agnostic_nms=True, verbose=False)[0]
    encontrados = []
    if r.masks is not None and len(r.boxes):
        for poly, cid, box in zip(r.masks.xyn, r.boxes.cls.cpu().numpy().astype(int), r.boxes.xyxy.cpu().numpy()):
            cls_name = PROMPTS_M2.get(r.names[cid])
            if cls_name and len(poly) >= 3:
                encontrados.append((ID_OBJETIVO[cls_name], poly, box))
    if filtro_cajas is None:
        return [(c, p) for c, p, _ in encontrados]
    # Empareja con las clases bolivianas reales (ej. minibus/micro)
    res = []
    for gt_cls_id, (gx1, gy1, gx2, gy2) in filtro_cajas:
        gcx, gcy = (gx1 + gx2) / 2 * W, (gy1 + gy2) / 2 * H
        mejor_poly = None
        for _, poly, (px1, py1, px2, py2) in encontrados:
            if px1 <= gcx <= px2 and py1 <= gcy <= py2:
                mejor_poly = poly
                break
        if mejor_poly is None:
            # Poligono limpio de respaldo a partir de la caja anotada del dataset boliviano
            x1, y1 = max(0.0, gx1), max(0.0, gy1)
            x2, y2 = min(1.0, gx2), min(1.0, gy2)
            mejor_poly = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=float)
        res.append((gt_cls_id, mejor_poly))
    return res

for ds_bo in (ds_bo1, ds_bo2):
    names_bo = cargar_names(ds_bo.location)
    for split_in in ('train', 'valid', 'test'):
        img_dir = Path(ds_bo.location) / split_in / 'images'
        if not img_dir.exists():
            continue
        sp_out = 'valid' if split_in in ('valid', 'test') else 'train'
        for img_p in sorted(img_dir.glob('*')):
            lbl_p = img_p.parent.parent / 'labels' / f'{img_p.stem}.txt'
            if not lbl_p.exists():
                continue
            gt_boxes = []
            for line in lbl_p.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                raw_cls = str(names_bo[int(float(parts[0]))]).lower()
                norm_cls = MAPA_SINONIMOS.get(raw_cls)
                if norm_cls is None:
                    continue
                cx, cy, bw, bh = [float(v) for v in parts[1:5]]
                gt_boxes.append((ID_OBJETIVO[norm_cls], (cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2)))
            if gt_boxes:
                im = cv2.imread(str(img_p))
                if im is not None:
                    polis = extraer_polis_yoloe(im, gt_boxes)
                    if polis:
                        registros.append((img_p, sp_out, polis))

# C) Desde Wikimedia Commons
random.seed(42)
for i, (img_p, _) in enumerate(wiki_imgs):
    im = cv2.imread(str(img_p))
    if im is None:
        continue
    polis = extraer_polis_yoloe(im)
    if polis:
        registros.append((img_p, 'valid' if i % 5 == 0 else 'train', polis))

print('Total de imagenes unificadas con poligonos:', len(registros))

## 3. Exportar YOLO-seg + COCO JSON, subir a Roboflow y entrenar YOLO11s-seg

In [ ]:
# Guardar dataset YOLO local + COCO JSON para subir a Roboflow
coco_splits = {
    sp: {
        'images': [],
        'annotations': [],
        'categories': [{'id': i, 'name': c, 'supercategory': 'vehiculo'} for i, c in enumerate(CLASES_OBJETIVO)],
    }
    for sp in ('train', 'valid')
}
ann_id = 1
conteo_clases = {c: 0 for c in CLASES_OBJETIVO}

for idx, (src_p, sp, polis) in enumerate(registros):
    im = cv2.imread(str(src_p))
    if im is None:
        continue
    H, W = im.shape[:2]
    fname = f'{idx:05d}_{src_p.stem}.jpg'
    cv2.imwrite(str(MERGED_YOLO / sp / 'images' / fname), im)
    cv2.imwrite(str(MERGED_COCO / sp / fname), im)

    coco_splits[sp]['images'].append({'id': idx + 1, 'file_name': fname, 'width': W, 'height': H})
    yolo_lines = []
    for cid, poly in polis:
        poly = np.clip(poly, 0.0, 1.0)
        if len(poly) < 3:
            continue
        conteo_clases[CLASES_OBJETIVO[cid]] += 1
        yolo_lines.append(f'{cid} ' + ' '.join(f'{x:.6f} {y:.6f}' for x, y in poly))
        px = poly[:, 0] * W
        py = poly[:, 1] * H
        seg_flat = [round(float(v), 2) for pt in np.column_stack([px, py]) for v in pt]
        x1, y1, x2, y2 = float(px.min()), float(py.min()), float(px.max()), float(py.max())
        bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
        coco_splits[sp]['annotations'].append({
            'id': ann_id, 'image_id': idx + 1, 'category_id': cid,
            'segmentation': [seg_flat], 'area': round(bw * bh, 2),
            'bbox': [round(x1, 2), round(y1, 2), round(bw, 2), round(bh, 2)], 'iscrowd': 0,
        })
        ann_id += 1
    (MERGED_YOLO / sp / 'labels' / f'{idx:05d}_{src_p.stem}.txt').write_text('\n'.join(yolo_lines))

for sp in ('train', 'valid'):
    (MERGED_COCO / sp / '_annotations.coco.json').write_text(json.dumps(coco_splits[sp]))

yaml.safe_dump({
    'path': str(MERGED_YOLO), 'train': 'train/images', 'val': 'valid/images',
    'nc': len(CLASES_OBJETIVO), 'names': {i: c for i, c in enumerate(CLASES_OBJETIVO)},
}, open(MERGED_YOLO / 'data.yaml', 'w'), sort_keys=False)

print('Instancias por clase:', conteo_clases)

# Subir al proyecto propio en Roboflow y generar version con aumentaciones
try:
    ws = rf.workspace(WORKSPACE)
    ws.upload_dataset(
        dataset_path=str(MERGED_COCO),
        project_name=PROJECT_M2,
        project_type='instance-segmentation',
        num_workers=8,
    )
    proj = ws.project(PROJECT_M2)
    ver_num = proj.generate_version(settings={
        'preprocessing': {'auto-orient': True, 'resize': {'width': 640, 'height': 640, 'format': 'Stretch to'}},
        'augmentation': {
            'image': {'versions': 2},
            'flip': {'horizontal': True, 'vertical': False},
            'brightness': {'brighten': True, 'darken': True, 'percent': 15},
            'rotation': {'degrees': 10},
        },
    })
    print('Subido a Roboflow y generada version:', ver_num)
    ds_rf = proj.version(int(ver_num)).download('yolov11', location='/content/m2_roboflow_final')
    DATA_YAML_M2 = str(Path(ds_rf.location) / 'data.yaml')
except Exception as exc:
    print('Aviso subida/versionado Roboflow (entrenando con copia local unificada):', exc)
    DATA_YAML_M2 = str(MERGED_YOLO / 'data.yaml')

In [ ]:
# Normaliza nombres por si Roboflow reordeno los IDs
cfg = yaml.safe_load(Path(DATA_YAML_M2).read_text())
print('Entrenando M2 con:', DATA_YAML_M2, '| Clases:', cfg.get('names'))

modelo_m2 = YOLO('yolo11s-seg.pt')
modelo_m2.train(
    data=DATA_YAML_M2,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project='/content/runs/segment',
    name='m2_vehiculos',
    exist_ok=True,
)

In [ ]:
BEST_M2 = Path('/content/runs/segment/m2_vehiculos/weights/best.pt')
val_res = YOLO(str(BEST_M2)).val(data=DATA_YAML_M2)
names_val = yaml.safe_load(Path(DATA_YAML_M2).read_text()).get('names', {})
if isinstance(names_val, list):
    names_val = {i: n for i, n in enumerate(names_val)}

metricas_m2 = {
    'modelo': 'yolo11s-seg',
    'total_imagenes_base': len(registros),
    'instancias_por_clase_base': conteo_clases,
    'box_map50': round(float(val_res.box.map50), 4),
    'box_map50_95': round(float(val_res.box.map), 4),
    'mask_map50': round(float(val_res.seg.map50), 4),
    'mask_map50_95': round(float(val_res.seg.map), 4),
    'mask_map50_por_clase': {
        str(names_val.get(i, i)): round(float(v), 4)
        for i, v in enumerate(val_res.seg.ap50)
    },
}
print(json.dumps(metricas_m2, indent=2, ensure_ascii=False))
Path('/content/resumen_metricas_m2.json').write_text(json.dumps(metricas_m2, indent=2, ensure_ascii=False))
shutil.copy(BEST_M2, '/content/best_m2_seg.pt')

import sys, platform
print('Diagnostico Colab M2:', json.dumps({
    'python': sys.version.split()[0], 'os': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))

from google.colab import files
files.download('/content/best_m2_seg.pt')
files.download('/content/resumen_metricas_m2.json')